# **Vaccination diphtérie-tétanos-coqueluche chez les enfants**
# Données issues des Enquêtes Démographiques et de Santé (EDS - DHS)


Ce rapport génère des visualisations pour l'indicateur relatif au **pourcentage d'enfants qui sont vaccinés avec le vaccin contre la diphtérie (D), le tétanos (T) et la coqueluche (P)**, à partir des données de l'Enquête Démographique et de Santé (EDS - DHS).

---

* *Numérateur* : Nombre d'enfants qui sont membres de fait du ménage, testés par test de diagnostic rapide (TDR) ou par microscopie, et dont le résultat est positif pour le paludisme.
* *Dénominateur* : Nombre d'enfants qui sont membres de fait du ménage, testés par test de diagnostic rapide (TDR) ou par microscopie.

---

En plus des indicateurs de vaccination pour chacune des trois doses de vaccin, le pipeline comprend aussi des indicateurs d'attrition (le pourcentage d'enfants ayant reçu la dose 1, mais pas la dose 2, etc.)

---

Pour plus d'informations (en anglais):
- Ressources relatives aux comportements de recherche de soins
    - [Définition et calculs](https://dhsprogram.com/data/Guide-to-DHS-Statistics/index.htm#t=Vaccination.htm%23Percentage_of_childrenbc-1&rhtocid=_13_1_0)
- [Les questionnaires utilisés dans les EDS/DHS](https://dhsprogram.com/publications/publication-dhsg4-dhs-questionnaires-and-manuals.cfm)

---

*Note* : Contrairement à la majorité des analyses dans le cadre du processus SNT, cette analyse est menée au niveau administratif **ADM1**, en raison de la disponibilité des données.

## 1. Configuration

In [ ]:
rm(list = ls())

options(scipen=999)

In [ ]:
# Global paths
Sys.setenv(PROJ_LIB = "/opt/conda/share/proj")
Sys.setenv(GDAL_DATA = "/opt/conda/share/gdal")

In [ ]:
# Paths
ROOT_PATH <- '~/workspace'
PIPELINE_PATH <- file.path(ROOT_PATH, 'pipelines', 'snt_dhs_indicators')
CONFIG_PATH <- file.path(ROOT_PATH, 'configuration')
CODE_PATH <- file.path(ROOT_PATH, 'code')
DATA_PATH <- file.path(ROOT_PATH, 'data')
DHS_DATA_PATH <- file.path(DATA_PATH, 'dhs', 'raw')
OUTPUT_DATA_PATH <- file.path(DATA_PATH, 'dhs', 'indicators', 'vaccination')
OUTPUT_PLOTS_PATH <- file.path(ROOT_PATH, 'pipelines', 'snt_dhs_indicators', 'reporting', 'outputs')

In [ ]:
# Load notebook-specific utilities
source(file.path(CODE_PATH, "snt_utils.r"))
source(file.path(CODE_PATH, "snt_report.r"))
source(file.path(CODE_PATH, "snt_palettes.r"))
source(file.path(PIPELINE_PATH, "utils", "snt_dhs_vaccination_report.r"))

# List required pcks
required_packages <- c("haven", "glue", "survey", "data.table", "sf", "ggplot2", "stringi", "reticulate", "jsonlite", "arrow", "IRdisplay")

# Execute function
install_and_load(required_packages)

In [ ]:
Sys.setenv(RETICULATE_PYTHON = "/opt/conda/bin/python")
reticulate::py_config()$python
openhexa <- import("openhexa.sdk")

# Load SNT config
CONFIG_FILE_NAME <- "SNT_config.json"
config_json <- tryCatch({ fromJSON(file.path(CONFIG_PATH, CONFIG_FILE_NAME)) },
                        error = function(e) {
                          msg <- paste0("Error while loading configuration", conditionMessage(e))  
                          cat(msg)   
                          stop(msg) 
                        })

msg <- paste0("SNT configuration : ", file.path(CONFIG_PATH, CONFIG_FILE_NAME)) 
log_msg(msg)

# Set config variables
COUNTRY_CODE <- config_json$SNT_CONFIG$COUNTRY_CODE

## 2. Chargement et pré-processing des données à visualiser

**Les données utilisées**

Toutes les données utilisées dans ce rapport sont agrégées au niveau administratif ADM1:

* Données spatiales : fond de carte (DHIS2)
* Données calculées par le pipeline :
    - valeurs estimées de vaccination DTP, ainsi que sur la précision statistique de cette estimation (intervalles de confiance à 95%)
    - niveaux estimés d'attrition pour la vaccination DTP

In [ ]:
admin_level <- 'ADM1'
admin_id_col <- glue(admin_level, 'ID', .sep='_')
admin_name_col <- glue(admin_level, 'NAME', .sep='_')
admin_cols <- c(admin_id_col, admin_name_col)

In [ ]:
# Load spatial file from dataset

dhis2_dataset <- config_json$SNT_DATASET_IDENTIFIERS$DHIS2_DATASET_FORMATTED

spatial_data_filename <- paste(COUNTRY_CODE, "shapes.geojson", sep = "_")
# spatial_data <- read_sf(file.path(DATA_PATH, 'dhis2', 'formatted', spatial_data_filename))
spatial_data <- get_latest_dataset_file_in_memory(dhis2_dataset, spatial_data_filename)
log_msg(glue("File {spatial_data_filename} successfully loaded from dataset version: {dhis2_dataset}"))

spatial_data <- st_as_sf(spatial_data)

# aggregate geometries by the admin columns
spatial_data <- aggregate_geometry(
  sf_data=spatial_data,
  admin_id_colname=admin_id_col,
  admin_name_colname=admin_name_col
  )

# keep class
spatial_data <- st_as_sf(spatial_data)

# DRC provinces need to be cleaned
if(COUNTRY_CODE == "COD"){
  spatial_data[[admin_name_col]] <- clean_admin_names(spatial_data[[admin_name_col]])
}

In [ ]:
data_source <- 'DHS'
vaccination_doses <- c(1, 2, 3)
indicator_access <- 'PCT_DTP'
indicator_attrition <- 'PCT_DROPOUT_DTP'

## 3. Création de graphiques

Le rapport crée deux types visualisations:
   - Une carte choroplèthe indiquant l'estimation moyenne de la valeur de l'indicateur, sur base des données d'enquête
   - Un graphique de son intervalle de confiance, avec des barres d’erreur pour chaque ADM1 (ceci n'est généré que pour les pourcentages d'enfants vaccinés, par dose du vaccin DTP; les indicateurs d'attrition se basant uniquement sur l'estimation ponctuelle de la moyenne, ces indicateurs sont présentés sans intervalles de confiance)

In [ ]:
# emtpy vectors to save locations of all files to render
dose_plot_paths <- character(0)
dose_ci_plot_paths <- character(0)
dropout_plot_paths <- character(0)

In [ ]:
for (dose_number in vaccination_doses){
  table_name <- glue("{toupper(indicator_access)}{dose_number}")
  filename_without_extension <- glue("{COUNTRY_CODE}_{data_source}_{admin_level}_{table_name}")
  df <- fread(file.path(OUTPUT_DATA_PATH, paste0(filename_without_extension, '.csv')))
    
  vaccine_colname <- glue("{toupper(indicator_access)}{dose_number}")
  
  # change the names of the columns
  sample_avg_col <- paste(vaccine_colname, 'SAMPLE_AVERAGE', sep = '_')
  lower_bound_col <- paste(vaccine_colname, 'CI_LOWER_BOUND', sep = '_')
  upper_bound_col <- paste(vaccine_colname, 'CI_UPPER_BOUND', sep = '_')
  
  # add spatial data
  plot_data <- merge(spatial_data, df, by = admin_cols, all = TRUE)
  
  # 1. MAPS

  # plot variables
  dose_plot_title = glue("Couverture vaccinale, DTP{dose_number} (%)")
  
  # make and save the plot
  dose_plot <- make_pct_choropleth_map(
    map_data = plot_data,
    target_colname = sample_avg_col,
    plot_title = dose_plot_title,
    plot_subtitle = COUNTRY_CODE,
    plot_caption = glue("Données: {data_source}")
    )
  
  dose_plot_filename <- glue("{COUNTRY_CODE}_{data_source}_{admin_level}_{toupper(indicator_access)}{dose_number}_plot.png")
  dose_plot_path = file.path(OUTPUT_PLOTS_PATH, dose_plot_filename)
  suppressMessages(ggsave(filename = dose_plot_path, plot = dose_plot, width = 6, height = 5, dpi = 300))
  
  dose_plot_paths <- c(dose_plot_paths, dose_plot_path) # add to be rendered later

  # 2. CI PLOTS

  # plot variables
  dose_ci_plot_title <- glue("Couverture vaccinale, DTP{dose_number} (Intervalles de confiance 95%)")
  dose_ci_plot_xlab <- admin_level
  dose_ci_plot_ylab <- glue("Enfants vaccinés DTP{dose_number} (%)")
  
  # make the confidence interval plot
  dose_ci_plot <- make_ci_plot(
    df_to_plot=plot_data,
    admin_colname=admin_name_col,
    point_estimation_colname=sample_avg_col,
    ci_lower_colname=lower_bound_col,
    ci_upper_colname=upper_bound_col,
    plot_title=dose_ci_plot_title,
    plot_subtitle=COUNTRY_CODE,
    plot_caption=glue("Données: {data_source}"),
    x_title=dose_ci_plot_xlab,
    y_title=dose_ci_plot_ylab
  )
  
  # save the ci plot
  dose_ci_plot_filename <- glue("{COUNTRY_CODE}_{data_source}_{admin_level}_{indicator_access}{dose_number}_CI_plot.png")
  dose_ci_plot_path <- file.path(OUTPUT_PLOTS_PATH, dose_ci_plot_filename)
  suppressMessages(ggsave(filename=dose_ci_plot_path, plot=dose_ci_plot, width = 6, height = 5, dpi = 300))

  dose_ci_plot_paths <- c(dose_ci_plot_paths, dose_ci_plot_path) # add to be rendered later

}

In [ ]:
# display all maps and confidence interval plots, avoiding lapply's NULL returns
invisible(lapply(dose_plot_paths, function(p) display_png(file = p)))
invisible(lapply(dose_ci_plot_paths, function(p) display_png(file = p)))

In [ ]:
dtp_dropout_filename_without_extension <- glue("{COUNTRY_CODE}_{data_source}_{admin_level}_{indicator_attrition}")
DTP_DROPOUT <- fread(file.path(OUTPUT_DATA_PATH, paste0(dtp_dropout_filename_without_extension, ".csv")))

In [ ]:
for(current_dose in vaccination_doses){
  for (reference_dose in 1:(current_dose - 1)){
    if((reference_dose >= 1) & (reference_dose < current_dose)){
    dropout_colname <- glue("{indicator_attrition}_{reference_dose}_{current_dose}")
   
    dropout_plot_title = glue("Attrition de DTP{reference_dose} à DTP{current_dose} (%)")
    dropout_plot_data <- merge(spatial_data, DTP_DROPOUT, by = admin_cols)
    dropout_plot <- make_pct_choropleth_map(
      map_data = dropout_plot_data,
      target_colname = dropout_colname,
      plot_title = dropout_plot_title,
      plot_subtitle = COUNTRY_CODE,
      plot_caption = glue("Données: {data_source}")
    )
      
    dropout_plot_filename <- glue("{COUNTRY_CODE}_{data_source}_{admin_level}_{toupper(dropout_colname)}_plot.png")
    dropout_plot_path <- file.path(OUTPUT_PLOTS_PATH, dropout_plot_filename)
    suppressMessages(ggsave(filename = dropout_plot_path, plot = dropout_plot, width = 6, height = 5, dpi = 300))

    dropout_plot_paths <- c(dropout_plot_paths, dropout_plot_path) # add to be rendered later
   
    }
  }
}

In [ ]:
# display all maps and confidence interval plots, avoiding lapply's NULL returns
invisible(lapply(dropout_plot_paths, function(p) display_png(file = p)))